In [1]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, MaxPooling2D, Dropout, Conv2D
from tensorflow.keras import optimizers

# Load and preprocess the license plate image
def preprocess_image(image_path):
    img = cv2.imread(image_path)
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, img_binary = cv2.threshold(img_gray, 200, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return img_binary

# Function to segment characters from license plate
def segment_characters(image):
    char_list = []
    contours, _ = cv2.findContours(image, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)[:15]

    for cntr in contours:
        x, y, w, h = cv2.boundingRect(cntr)
        if 10 < w < 100 and 20 < h < 100:  # Dimension constraints
            char = image[y:y+h, x:x+w]
            char = cv2.resize(char, (28, 28))
            char_list.append(char)

    return np.array(char_list)

# Load and preprocess the image
image_path = "car_plate.png"  # Change this to your image path
binary_image = preprocess_image(image_path)
characters = segment_characters(binary_image)

# Display segmented characters
plt.figure(figsize=(10, 3))
for i, ch in enumerate(characters):
    plt.subplot(1, len(characters), i+1)
    plt.imshow(ch, cmap='gray')
    plt.axis('off')
plt.show()

# Model setup
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(28, 28, 1), padding='same'),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.4),
    Flatten(),
    Dense(128, activation='relu'),
    Dense(36, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer=optimizers.Adam(learning_rate=0.0001), metrics=['accuracy'])

# Data loading
train_datagen = ImageDataGenerator(rescale=1./255)
train_generator = train_datagen.flow_from_directory(
    'data/train', target_size=(28,28), batch_size=1, class_mode='categorical')

validation_generator = train_datagen.flow_from_directory(
    'data/val', target_size=(28,28), batch_size=1, class_mode='categorical')

# Training the model (Reduced epochs to 10)
model.fit(train_generator, validation_data=validation_generator, epochs=1)

# Function to predict characters
def predict_characters(chars):
    dic = {i: c for i, c in enumerate('0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ')}
    output = []

    for ch in chars:
        img = cv2.resize(ch, (28,28))
        img = img.reshape(1,28,28,1) / 255.0
        pred = np.argmax(model.predict(img))
        output.append(dic[pred])

    return ''.join(output)

# Predict and print the license plate number
plate_number = predict_characters(characters)
print("Predicted License Plate:", plate_number)


ModuleNotFoundError: No module named 'tensorflow'